# Examine search trace (v1)

Step-by-step inspection of ONE search method on ONE question —
for debugging and understanding the search behavior, not for
measuring accuracy. Nothing is written to `results/` or W&B.

Workflow:

1. pick `METHOD` + config `OVERRIDES` (section 2),
2. load the models (section 3), pick the question (section 4)
   and the trace `VERBOSITY` (section 5),
3. run the search (section 6) — the live trace streams into
   the cell output,
4. inspect the run post-hoc: summary stats, the full search
   tree, completed nodes, per-step PRM re-score (sections
   7–10).

How it works: the live trace reuses the `logging.fatal` /
`logging.error` calls already inside the search cores (the
launchers silence them; this notebook re-enables the root
logger). The post-hoc tree view works because the notebook
builds the `MCTS` agent itself and keeps it after
`mcts_search` returns — no monkey-patching needed.

GPU required (vLLM engine + PRM).


## 1. Setup


In [1]:
import os

os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
sys.path.insert(0, "..")

import time
import random
import logging
import importlib

import numpy as np
import torch
from vllm import LLM
from hydra import initialize, compose
from hydra.core.config_store import ConfigStore
from omegaconf import OmegaConf

from core.reward_models import build_prm
from utils.configs import (
    ExpConfig, MCTSCntConfig, BLMCTSCntConfig,
    MCTSSemV01Config, MCTSSemV02Config,
)
from utils.load_data import load_data_hf
from notebook_utils import gpu_mem_used_gb, print_step_scores

assert torch.cuda.is_available(), "CUDA is required"
print(torch.cuda.get_device_name(0))


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu126).


Tesla V100S-PCIE-32GB


## 2. Pick the method and compose the config

`METHOD` selects the search core + its Hydra root config;
`OVERRIDES` is the same override list a launcher command would
take. Keep the budget tiny — the point is to watch the
mechanics, not to finish a real run.

Switching `METHOD` only re-imports the core module; the loaded
LLM/PRM are reusable as long as the `llm=`/`prm=` groups are
unchanged (otherwise re-run section 3, which needs a kernel
restart to free GPU memory first).


In [2]:
# One entry per search variant. "embeds" marks the extra
# llm_vllm_embeds argument that only the sem family takes.
METHODS = {
    "mcts_cnt": {
        "module": "core.mcts_cnt_search_v01_00_00",
        "config_name": "mcts_cnt_prm800k",
        "embeds": False,
    },
    "mcts_bl_cnt_v01": {
        "module": "core.mcts_bl_cnt_search_v01_00_00",
        "config_name": "mcts_bl_cnt_v01_prm800k",
        "embeds": False,
    },
    "mcts_sem_v02": {
        "module": "core.mcts_sem_search_v02_00_00",
        "config_name": "mcts_sem_v02_prm800k",
        "embeds": True,
    },
}
METHOD = "mcts_cnt"

# Small budget => a trace short enough to read end to end but
# large enough to usually reach a completion (llama-1b solutions
# run ~10+ steps; see section 6 note on zero completions).
OVERRIDES = [
    "llm=llama_1b",
    "search.gen_budget=24",
]

# Register the structured schemas (all of them — harmless), same
# as the launchers do, so the YAML binds onto typed dataclasses.
cs = ConfigStore.instance()
cs.store(name="exp_schema", node=ExpConfig)
cs.store(group="search", name="mcts_cnt_schema", node=MCTSCntConfig)
cs.store(
    group="search", name="mcts_bl_cnt_v01_schema",
    node=BLMCTSCntConfig,
)
cs.store(
    group="search", name="mcts_sem_v01_schema",
    node=MCTSSemV01Config,
)
cs.store(
    group="search", name="mcts_sem_v02_schema",
    node=MCTSSemV02Config,
)

spec = METHODS[METHOD]
core_mod = importlib.import_module(spec["module"])

with initialize(version_base=None, config_path="../conf"):
    cfg = compose(config_name=spec["config_name"], overrides=OVERRIDES)

print(f"method = {METHOD}  ({spec['module']})")
print(OmegaConf.to_yaml(cfg.search))


ANTLR runtime and generated code versions disagree: 4.7.2!=4.9.3
ANTLR runtime and generated code versions disagree: 4.7.2!=4.9.3
ANTLR runtime and generated code versions disagree: 4.7.2!=4.9.3
ANTLR runtime and generated code versions disagree: 4.7.2!=4.9.3
ANTLR runtime and generated code versions disagree: 4.7.2!=4.9.3
ANTLR runtime and generated code versions disagree: 4.7.2!=4.9.3
method = mcts_cnt  (core.mcts_cnt_search_v01_00_00)
method: mcts_cnt_v01
batch_size: 4
lookahead: 0
max_depth: 20
negative_reward: 0.0
num_phases: 1000
gen_budget: 24
cpuct: 2.0
prm_batch_size: 1



## 3. Load the models

Mirrors the launcher exactly (incl. `quantization` /
`load_format` / `seed`). The second pooling engine is built only
for the sem-v01 policy-embeds path; sem-v02 pulls embeds from
the PRM forward pass and receives `None`.


In [3]:
llm_vllm = LLM(
    model=cfg.llm.llm_dir,
    tensor_parallel_size=cfg.llm.tensor_parallel_size,
    max_model_len=cfg.llm.max_model_len,
    gpu_memory_utilization=cfg.llm.gpu_memory_utilization,
    enforce_eager=cfg.llm.enforce_eager,
    distributed_executor_backend=None,
    dtype=cfg.llm.dtype,
    quantization=cfg.llm.quantization,
    load_format=cfg.llm.load_format,
    seed=cfg.gen.seed,
)

llm_vllm_embeds = None
if spec["embeds"] and cfg.search.embeds_source == "policy":
    llm_vllm_embeds = LLM(
        model=cfg.llm.llm_dir,
        runner="pooling",
        tensor_parallel_size=cfg.llm.tensor_parallel_size,
        max_model_len=cfg.llm.max_model_len,
        gpu_memory_utilization=(
            cfg.search.embeds_gpu_memory_utilization
        ),
        enforce_eager=cfg.llm.enforce_eager,
        distributed_executor_backend=None,
        dtype=cfg.llm.dtype,
        seed=cfg.gen.seed,
    )

prm = build_prm(
    cfg.prm.kind, cfg.prm.prm_dir, device=cfg.prm.device_map,
)
print(f"GPU mem used: {gpu_mem_used_gb():0.2f} GB")


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu126).


Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8


<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.83s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.83s/it]



Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):   8%|▊         | 4/51 [00:00<00:01, 36.52it/s]

...

Capturing CUDA graphs (decode, FULL):  49%|████▊     | 17/35 [00:00<00:00, 53.50it/s]

Capturing CUDA graphs (decode, FULL):  86%|████████▌ | 30/35 [00:00<00:00, 58.41it/s]

Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:00<00:00, 52.72it/s]


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

GPU mem used: 25.62 GB


## 4. Pick the question


In [4]:
load_kwargs = {"ds_split": cfg.data.ds_split}
if cfg.data.level is not None:
    load_kwargs["level"] = cfg.data.level
dataset = load_data_hf(cfg.data.ds_dir, **load_kwargs)

QUESTION_IDX = 0
record = dataset[QUESTION_IDX]
question = record[cfg.data.question_field]

print(f"{len(dataset)} questions (level {cfg.data.level})")
print(f"--- question {QUESTION_IDX} ---")
print(question)
for key in ("answer", "solution"):
    if key in record:
        print(f"--- reference {key} ---")
        print(record[key])
        break


128 questions (level 4)
--- question 0 ---
The set of points $(x,y,z)$ that satisfy
\[2x = 3y = -z\]is a line.

The set of points $(x,y,z)$ that satisfy
\[6x = -y = -4z\]is another line.

Find the angle between these lines, in degrees.
--- reference answer ---
90^\circ


## 5. Trace verbosity

The search cores log their internals on two levels:

- `FATAL` (50) — per-phase **selection** trace: PUCT breakdown
  per candidate node (q / u / visits / terminal), the selected
  node, running `gen_cnt`;
- `ERROR` (40) — additionally the **generation** details:
  current templated text, raw `generate_k_steps` outputs
  (incl. `stop_reasons`), per-candidate PRM scores.

The modules set the root logger *above* FATAL at import (that
is what keeps launchers silent); this cell just lowers it back.
Re-run this cell anytime to change verbosity between runs.


In [5]:
VERBOSITY = "selection"  # "silent" | "selection" | "generation"

_LEVELS = {
    "silent": logging.FATAL + 1,
    "selection": logging.FATAL,
    "generation": logging.ERROR,
}
root_logger = logging.getLogger()
if not root_logger.handlers:
    logging.basicConfig(format="%(message)s")
root_logger.setLevel(_LEVELS[VERBOSITY])
print(f"trace verbosity: {VERBOSITY}")


trace verbosity: selection


## 6. Run the search

Seeding mirrors `_search`'s per-question seed
(`100_000 + trial_idx`), so with the same config this run is
comparable with trial `TRIAL_IDX` of a recorded run.

Note: with a tiny `gen_budget`, ending with **zero completions
is common and not a bug** — especially for `mcts_bl_cnt_v01`
(see `docs/findings/coding-findings/`
`bl-cnt-frontier-zero-completion-rate.md`).


In [6]:
TRIAL_IDX = 0

seed = 100_000 + TRIAL_IDX
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

agent = core_mod.MCTS(config=cfg, question=question)

search_args = [question, agent, cfg, llm_vllm]
if spec["embeds"]:
    search_args.append(llm_vllm_embeds)
search_args.append(prm)

start = time.time()
(
    completions, comp_depth, comp_phase, comp_gen,
    q_total_gens, q_last_phase, phase_depths,
    q_nodes_max_depth,
) = core_mod.mcts_search(*search_args)
print(f"\nsearch took {time.time() - start:0.1f}s")



-> p = 0



-> d = 0


0.1


   q-value = 0.0706


   u-value = 0.0000


...

   is_terminal = False


selected_child = 0.3.4.1.1.3.4.4.3.2


gen_cnt = 24


run out of budget!



search took 34.4s


## 7. Run summary

The same per-question fields a recorded
`generate_...--trial-XXX.jsonl` would hold.


In [7]:
print(f"generations used   : {q_total_gens}/{cfg.search.gen_budget}")
print(f"last phase reached : {q_last_phase}")
print(f"phase_depths       : {phase_depths}")
print(f"nodes at max depth : {q_nodes_max_depth}")
print(f"completions        : {len(completions)}")
for i, (dep, ph, gc) in enumerate(
    zip(comp_depth, comp_phase, comp_gen)
):
    print(f"  [{i}] depth={dep}  phase={ph}  gen_cnt={gc}")


generations used   : 24/24
last phase reached : 1
phase_depths       : [16, 8]
nodes at max depth : 0
completions        : 2
  [0] depth=12  phase=0  gen_cnt=12
  [1] depth=16  phase=0  gen_cnt=16


## 8. Search tree

One node per line: dotted `tag` (lineage), the stats the
selection rule actually uses (`n`=visits, `q`=mean value), the
flags that decide a node's fate (`T`=terminal, `C`=completed),
and a preview of the step text this node added on top of its
parent.


In [8]:
def node_step_text(node):
    """The step this node added on top of its parent."""
    text = node.state["text"]
    if node.parent is not None:
        text = text.removeprefix(node.parent.state["text"])
    return text


def print_tree(node, max_chars=60, _prefix=""):
    """ASCII dump of the search tree, one node per line."""
    step = node_step_text(node).replace("\n", "\\n")
    if len(step) > max_chars:
        step = step[:max_chars] + "..."
    flags = ""
    if node.is_terminal:
        flags += " T"
    if node.is_completed:
        flags += " C"
    print(
        f"{_prefix}{node.tag}  "
        f"n={node.visit_count()} q={node.q_value():0.3f}"
        f"{flags}  |{step}|"
    )
    for child in node.children:
        print_tree(child, max_chars=max_chars, _prefix=_prefix + "  ")


print_tree(agent.root)


0  n=2 q=0.477  ||
  0.1  n=1 q=0.071  |## Step 1: The direction vectors of the lines can be obtaine...|
  0.2  n=2 q=0.853  |## Step 1: Identify the equations of the lines\nThe first eq...|
    0.2.1  n=1 q=0.615  |## Step 2: Identify the equations of the second line\nThe se...|
    0.2.2  n=2 q=0.882  |## Step 2: Identify the second equation\nThe second equation...|
      0.2.2.1  n=1 q=0.932  |## Step 3: Calculate the angle between the lines\nTo find th...|
      0.2.2.2  n=2 q=0.948  |## Step 3: Calculate the angle between the lines using the d...|
        0.2.2.2.1  n=1 q=0.973  |## Step 4: Simplify the expression\nWe can simplify the expr...|
        0.2.2.2.2  n=1 q=0.978  |## Step 4: Simplify the expression for $\cos \theta$\nEvalua...|
        0.2.2.2.3  n=2 q=0.967  |## Step 4: Simplify the expression\nSimplify the numerator: ...|
          0.2.2.2.3.1  n=1 q=0.434  |## Step 5: Evaluate $\cos \theta$\n$\cos \theta = \frac{13}{...|
          0.2.2.2.3.2  n=1 q=0.808  |## Step 

## 9. Completed nodes

Full text of every EOS/length-terminated leaf (the pool that
`completions` is deduped from), with the tree coordinates and
the node's final q-value.


In [9]:
if not agent.completed_nodes:
    print("no completed nodes (budget exhausted before any "
          "EOS/length stop)")
for node in agent.completed_nodes:
    text = node.state["text"]
    print(f"=== node {node.tag}  depth={node.depth} "
          f"phase={node.phase} gen_cnt={node.gen_cnt} "
          f"q={node.q_value():0.3f} ===")
    print(text if len(text) <= 1500 else text[:1500] + "\n[...]")
    print()


=== node 0.2.2.2.3.3.1.4.3.4.4.1.3  depth=12 phase=0 gen_cnt=12 q=0.523 ===
## Step 1: Identify the equations of the lines
The first equation is $2x = 3y = -z$, which can be written as $\frac{x}{2} = \frac{y}{3} = \frac{-z}{1}$. This gives the direction ratios of the first line as (2, 3, -1).

## Step 2: Identify the second equation
The second equation is $6x = -y = -4z$, which can be written as $\frac{x}{6} = \frac{-y}{1} = \frac{-4z}{1}$. This gives the direction ratios of the second line as (6, -1, -4).

## Step 3: Calculate the angle between the lines using the dot product formula
The angle $\theta$ between two lines with direction ratios (a, b, c) and (d, e, f) is given by the formula $\cos \theta = \frac{ad + be + cf}{\sqrt{a^2 + b^2 + c^2} \sqrt{d^2 + e^2 + f^2}}$. Applying this formula with (a, b, c) = (2, 3, -1) and (d, e, f) = (6, -1, -4) yields $\cos \theta = \frac{(2)(6) + (3)(-1) + (-1)(-4)}{\sqrt{(2)^2 + (3)^2 + (-1)^2} \sqrt{(6)^2 + (-1)^2 + (-4)^2}}$.

## Step 4: Simpli

## 10. Per-step PRM re-score

Re-score one completion step by step — handy for checking how a
node's aggregated score came about. Step/score counts can
differ by one because `PRM.score` splits a bogus trailing empty
step (see `docs/findings/coding-findings/`
`prm-step-split-trailing-separator.md`).


In [10]:
if completions:
    COMP_IDX = 0
    text = completions[COMP_IDX]
    steps = [s for s in text.split("\n\n") if s.strip()]
    scores = prm.score([question], [[text]])[0][0]
    print(f"{len(steps)} steps, {len(scores)} PRM scores")
    agg = core_mod.aggregate_scores(scores, cfg.gen.agg_strategy)
    print(f"aggregated ({cfg.gen.agg_strategy}): {agg:0.4f}\n")
    print_step_scores(steps, scores)
else:
    print("no completions to score")


12 steps, 12 PRM scores
aggregated (last): 0.5234

Step 1: P(correct) = 0.7520
## Step 1: Identify the equations of the lines
The first equ...
Step 2: P(correct) = 0.8105
## Step 2: Identify the second equation
The second equation ...
Step 3: P(correct) = 0.9443
## Step 3: Calculate the angle between the lines using the d...
Step 4: P(correct) = 0.9805
## Step 4: Simplify the expression
Simplify the numerator: $...
Step 5: P(correct) = 0.8843
## Step 5: Calculate the value of $\cos \theta$
Therefore, $...
Step 6: P(correct) = 0.8311
## Step 6: Calculate the value of $\theta$
Now, let's calcul...
Step 7: P(correct) = 0.8687
## Step 7: Convert the angle from radians to degrees
To conv...
Step 8: P(correct) = 0.8931
## Step 8: Calculate the numerical value of $\theta$ in degr...
Step 9: P(correct) = 0.9111
## Step 9: Simplify the numerical value of $\theta$ in degre...
Step 10: P(correct) = 0.9126
## Step 10: Use a calculator to evaluate the expression
Usin...
Step 11: P(correct) = 0.8887